# Multi-Modal RAG with Image & Table Captioning: Fusion approach
## Technique used: **Docling, Vision Transformer, FAISS, BM25, Reranker, CrossEncoder**

## Overview

Standard RAG only processes **text**. But many documents (research papers, reports) contain **images, figures, and tables** that hold critical information. Multi-modal RAG extracts and processes both.

| Standard RAG | Multi-Modal RAG |
|---|---|
| Extract text only | Extract text **+ images+tables** |
| Images are ignored | Images are **captioned by a vision model** |
| Tables are truncated as regular text | Tables are **converted to Markdown** => easier to retrieve |
| Vector store has text chunks | Vector store has text chunks **+ image captions/image summary + table captions/table content/table summary** |

## Pipeline

1. Extract text, images and tables from a PDF using Docling package
2. Use a **vision model** to generate summary for each image.
3. Image caption will be merged with image content
4. Tables are converted to markdown and merged with Table caption
5. Chunk text, image captions/image summary, table captions/table content
6. Store everything in a single vector store using FAISS and BM25
7. Retrieval using fusion approach for top 10 chunks with embeded query:
    - Semantic search using FAISS 
    - Keyword search using BM25
8. Apply Reranker using Cross Encoder to rerank both top chunks, shortlist to 5 chunks
9. Feed these 5 chunks and query to LLM to return answer

## Models Used

- **Vision LLM**: `qwen3-vl:8b` via Ollama (image captioning)
- **Text LLM**: `qwen3:14b` via Ollama (answer generation)
- **Embeddings**: `qwen3-embedding:4b` via Ollama
- **Cross Encoder:** `ms-marco-MiniLM-L-6-v2` via HuggingFace

---
## Step 0: Import Packages

In [1]:
%%time
from PIL import Image
import io,os,shutil
import ollama, base64, string
import numpy as np 
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_ollama import ChatOllama

from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
from docling.document_converter import PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
import logging
import onnxruntime as ort

from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
from tqdm import tqdm
from IPython.display import display, HTML,Markdown


/users/tuev/.conda/envs/langchain/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


CPU times: user 7.01 s, sys: 4.69 s, total: 11.7 s
Wall time: 38.4 s


## Step 1: Use Docling to read PDF

In [2]:
logging.getLogger("RapidOCR").setLevel(logging.WARNING)
ort.set_default_logger_severity(4)  # hide non-fatal ONNX Runtime logs

In [3]:
%%time
pipeline_options = PdfPipelineOptions()
pipeline_options.images_scale = 2.0        # higher resolution for image extraction
pipeline_options.generate_page_images = True
pipeline_options.generate_picture_images = True
pipeline_options.generate_table_images = True

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

source = "data/attention_is_all_you_need.pdf"
result = converter.convert(source)

[INFO] 2026-02-27 17:17:43,916 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-02-27 17:17:43,936 [RapidOCR] download_file.py:60: File exists and is valid: /work/projects/tuev/LLMs/LLMs/conda_env/langchain/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-02-27 17:17:43,937 [RapidOCR] main.py:53: Using /work/projects/tuev/LLMs/LLMs/conda_env/langchain/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-02-27 17:17:43,997 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-02-27 17:17:44,003 [RapidOCR] download_file.py:60: File exists and is valid: /work/projects/tuev/LLMs/LLMs/conda_env/langchain/lib/python3.10/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-02-27 17:17:44,003 [RapidOCR] main.py:53: Using /work/projects/tuev/LLMs/LLMs/conda_env/langchain/lib/python3.10/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-02-27 17:17:44

CPU times: user 6min 33s, sys: 1.96 s, total: 6min 35s
Wall time: 51.5 s


## Step 2: Split into regular text, table and images

In [4]:
dir_path = "extracted_images"

# Remove directory and all existing contents if it exists
if os.path.exists(dir_path):
    shutil.rmtree(dir_path)

# Recreate the empty directory
os.makedirs(dir_path, exist_ok=True)

In [5]:
from docling.datamodel.document import TableItem, PictureItem, TextItem, SectionHeaderItem

text_data = []
table_data = []
image_data = []

last_item_was_picture = False
last_item_was_table = False

for item, _level in result.document.iterate_items():
    if isinstance(item, TextItem):
        text_content = item.text
        if not text_content.strip():
            continue

        if last_item_was_picture:
            if image_data:
                image_data[-1]["response"] = text_content
            last_item_was_picture = False
        elif last_item_was_table:
            if table_data:
                table_data[-1]["caption"] = text_content
            last_item_was_table = False
        else:
            text_data.append({
                "response": text_content,
                "name": f"text_{item.label}_{len(text_data)}",
                "type": "text"
            })

    elif isinstance(item, TableItem):
        last_item_was_picture = False
        last_item_was_table = True
        table_md = item.export_to_markdown()
        table_data.append({
            "response": table_md,
            "caption": "",
            "name": f"table_{len(table_data)+1}",
            "type": "table"
        })

    elif isinstance(item, PictureItem):
        last_item_was_picture = True
        last_item_was_table = False
        image_path = f"extracted_images/figure_{len(image_data)+1}.png"
        if item.image:
            item.image.pil_image.save(image_path)
            image_data.append({
                "response": "",
                "name": f"figure_{len(image_data)+1}.png",
                "type": "image"
            })

Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.


## Step 3: Caption Images with Vision Model
- In this section we use **"qwen3-vl:8b"** to read and summarize the image

In [6]:
caption_prompt = (
    "You are an assistant tasked with summarizing images for retrieval. "
    "These summaries will be embedded and used to retrieve the raw text. "
    "Give a concise summary of the text that is well optimized for retrieval. "
    "image:"
)

image_files = sorted([f for f in os.listdir("extracted_images") if not f.startswith(".")])
for img_name in image_files:
    img_path = f"extracted_images/{img_name}"

    response = ollama.chat(
        model="qwen3-vl:8b",
        messages=[{
            "role": "user",
            "content": caption_prompt,
            "images": [img_path],
        }],
    )

    caption = response.message.content
    image_data.append({"response": caption, "name": img_name})
    print(f"  {img_name}: {caption[:100]}...")

  figure_1.png: Transformer architecture diagram: Left side shows encoder with Input Embedding + Positional Encoding...
  figure_2.png: Attention mechanism operations sequence: MatMul(Q,K) → Scale → Mask (optional) → SoftMax → MatMul(·,...
  figure_3.png: Multi-Head Attention architecture featuring multiple parallel Scaled Dot-Product Attention heads, ea...
  figure_4.png: A majority of American governments have passed new laws since 2009 making the registration or voting...
  figure_5.png: The Law will never be perfect, but its application should be just. This is what we are missing in my...
  figure_6.png: The Law will never be perfect but its application should be just this is what we are missing in my o...


- Merge Image caption with Image summary

In [7]:
merged = {}
for d in image_data:
    name = d['name']
    if name not in merged:
        merged[name] = {**d, 'response': [d['response']]}
    else:
        merged[name]['response'].append(d['response'])
        for k, v in d.items():
            if k not in ('name', 'response') and k not in merged[name]:
                merged[name][k] = v

image_list = list(merged.values())

## Step 4: Summarize Tables for Better Retrieval
- We use regular LLM in this case **qwen3:14b** to summarize the table content

In [8]:
for tbl in table_data:
    response = ollama.chat(
        model="qwen3:14b",
        messages=[{
            "role": "user",
            "content": (
                "Summarize this table in 2-3 sentences for search retrieval. "
                "Then include the full table.\n\n" + tbl["response"]
            ),
        }],
    )
    tbl["summary"] = response.message.content

## Step 5: Create list of docs, table and images 
- Create docs_list: contains list of text document
- Create table_list: contains list of table
- Create img_list: contains list of text image caption and summary

In [9]:
# Text corpus — chunk it
docs_list = []
for t in text_data:
    doc = Document(
        page_content=t["response"],
        metadata={"name": t["name"], "type": "text"}
    )
    docs_list.append(doc)


In [10]:
# Table corpus — keep each table as a single chunk (don't split!)
table_list = []
for t in table_data:
    doc = Document(        
        page_content=t["caption"]+ "\n" +t["response"],
        metadata={"name": t["name"], "type": "table", "full_table": t["response"]}
    )
    table_list.append(doc)



In [11]:
display(Markdown(table_list[3].page_content))

Table 4: The Transformer generalizes well to English constituency parsing (Results are on Section 23 of WSJ)
| Parser                             | Training                 |   WSJ 23 F1 |
|------------------------------------|--------------------------|-------------|
| Vinyals &Kaiser el al. (2014) [37] | WSJ only, discriminative |        88.3 |
| Petrov et al. (2006) [29]          | WSJ only, discriminative |        90.4 |
| Zhu et al. (2013) [40]             | WSJ only, discriminative |        90.4 |
| Dyer et al. (2016) [8]             | WSJ only, discriminative |        91.7 |
| Transformer (4 layers)             | WSJ only, discriminative |        91.3 |
| Zhu et al. (2013) [40]             | semi-supervised          |        91.3 |
| Huang &Harper (2009) [14]          | semi-supervised          |        91.3 |
| McClosky et al. (2006) [26]        | semi-supervised          |        92.1 |
| Vinyals &Kaiser el al. (2014) [37] | semi-supervised          |        92.1 |
| Transformer (4 layers)             | semi-supervised          |        92.7 |
| Luong et al. (2015) [23]           | multi-task               |        93   |
| Dyer et al. (2016) [8]             | generative               |        93.3 |

In [12]:
# Image corpus — chunk captions (same as L17)
img_list = []
for i in image_list:
    doc = Document(
        page_content='\n'.join(i["response"]),
        metadata={"name": i["name"], "type": "image"}
    )
    img_list.append(doc)

## Step 6. Embed all data

In [13]:
embedding_model = OllamaEmbeddings(model="qwen3-embedding:4b")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=100
)

doc_splits = text_splitter.split_documents(docs_list)


In [14]:
# Tables and images stay whole — don't split them
all_chunks = img_list + table_list + doc_splits

In [15]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(all_chunks, embedding_model)

## Step 7. Retrieval
- We will use fusion approach
- Get top 5 search from BM25 using keyword search
- Get top 5 search from FAISS using semantic search


In [16]:
query = "What law attention to in Figure 4?"

### BM25

BM25 (Best Matching 25) is a classic keyword-based ranking function. It scores documents by how often the query terms appear, adjusted for document length. We build the BM25 index from the same chunks used in the vector store.

In [17]:
def bm25_tokenizer(text):
    tokenized_doc = []
    for token in text.lower().split():
        token = token.strip(string.punctuation)

        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)
    return tokenized_doc

In [18]:
tokenized_corpus = []
for chunk in tqdm(all_chunks):
    tokenized_corpus.append(bm25_tokenizer(chunk.page_content))

bm25 = BM25Okapi(tokenized_corpus)

100%|██████████████████████████████████████████████████████████████████████████████████████| 164/164 [00:00<00:00, 56239.54it/s]


In [19]:
bm25_scores = bm25.get_scores(bm25_tokenizer(query))

In [20]:
top_k = 10
top_bm25_k = np.argsort(bm25_scores)[::-1][:top_k]


In [21]:
Top_BM25_docs = [all_chunks[i] for i in top_bm25_k]

### Semantic Search

In [22]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": top_k})

In [23]:
Top_faiss_docs = retriever.invoke(query)


## Reranker
- Merge Semantic retrieval and Keyword retrieval
- Apply Cross Encoder technique to select the best 3 chunks

In [24]:
Merge2 = Top_BM25_docs+Top_faiss_docs

In [25]:
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Initial retrieval
initial_k = 10
rerank_top_k = 5

# Score each (query, document) pair with the Cross-Encoder
pairs = [[query, doc.page_content] for doc in Merge2]
ce_scores = cross_encoder.predict(pairs)

print("\nCross-Encoder scores:")
for i, (doc, score) in enumerate(zip(Merge2, ce_scores)):
    print(f"  Doc {i+1}: score = {score:.4f}  |  {doc.page_content[:80]}...")

# Sort by score descending, keep top-k
scored_pairs = sorted(zip(ce_scores, Merge2), key=lambda x: x[0], reverse=True)
reranked_docs_ce = [doc for _, doc in scored_pairs[:rerank_top_k]]

print(f"\nTop {rerank_top_k} after Cross-Encoder reranking:")
for i, doc in enumerate(reranked_docs_ce):
    print(f"  Doc {i+1}: {doc.page_content[:150]}...")


Cross-Encoder scores:
  Doc 1: score = 5.7527  |  Figure 4: Two attention heads, also in layer 5 of 6, apparently involved in anap...
  Doc 2: score = 2.4655  |  Figure 5: Many of the attention heads exhibit behaviour that seems related to th...
  Doc 3: score = -8.1685  |  4 Why Self-Attention...
  Doc 4: score = -2.8865  |  Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention cons...
  Doc 5: score = -9.0817  |  output values. These are concatenated and once again projected, resulting in the...
  Doc 6: score = -1.7275  |  We call our particular attention "Scaled Dot-Product Attention" (Figure 2). The ...
  Doc 7: score = -4.4818  |  Similarly, self-attention layers in the decoder allow each position in the decod...
  Doc 8: score = 1.0261  |  Figure 3: An example of the attention mechanism following long-distance dependen...
  Doc 9: score = -6.2959  |  The Transformer follows this overall architecture using stacked self-attention a...
  Doc 10: score = -3.51

---
## Step 5: Query and Generate an Answer

We ask a question about a **figure** in the paper. The retriever should find the relevant image caption, and the LLM generates an answer from it.

In [26]:
# Generate answer
llm = ChatOllama(model="qwen3:14b", temperature=0)

answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an assistant for question-answering tasks. Answer the question based upon your knowledge. "
     "Use three-to-five sentences maximum and keep the answer concise."),
    ("human",
     "Retrieved documents: \n\n <docs>{documents}</docs> \n\n User question: <question>{question}</question>"),
])


In [27]:

answer_chain = answer_prompt | llm | StrOutputParser()
answer = answer_chain.invoke({"documents": reranked_docs_ce, "question": query})

print(f"\nAnswer: {answer}")


Answer: Figure 4 discusses attention heads in layer 5 of 6, focusing on the phrase "The Law will never be perfect, but its application should be just." The attention mechanism highlights how the word "its" relates back to "The Law," demonstrating anaphora resolution. The example illustrates the model's ability to connect pronouns with their antecedents in the text.


---
## Summary

| Step | What happened |
|---|---|
| 1 | Downloaded the "Attention Is All You Need" paper |
| 2 | Extracted text, tables and images from the PDF |
| 3 | **Captioned each image** using `qwen3-vl:8b` vision model |
| 4 | Chunked both text, table and image with their corresponding captions, stored in vector store |
| 5 | Queried about a figure or table → retrieved the relevant docs |
| 6 | Using fusion approach of keyword and Semantic Similarity search to retrieve top 10 for each approach|
| 7 | From above 20 chunks, applied Reranker using Cross Encoder to retrieve final 5 |
| 8 | Return answer from final 5 chunks using LLM |

**Key insight:** By splitting text, tables and images with its captioning and storing those captions as searchable text, we make visual content (diagrams, tables, figures) retrievable through the same vector search pipeline as regular text. 